# Simulation Run (30 min Calibrated)

This script executes a 2 hour simulation of the model. The parameters of this model have already been calibrated to best-fit the 30 min AmpC fold induction data points from Uehara and Park (2002). An assortment of plots describing the time course of important variables are generated. Furthermore, two csv files, namely `uehara_park_ampc_summary.csv` and `uehara_park_fold_summary.csv` are created that relay model-generated AmpC concentration and fold change values, respectively, across the four experimental conditions along with literature-reported values.

# Import Statements

In [1]:
import tellurium as te
import roadrunner
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Directory Configs
PLOT_DIR = "plots"
os.makedirs(PLOT_DIR, exist_ok = True)

# Report only fatal errors 
roadrunner.Logger.setLevel(roadrunner.Logger.LOG_FATAL)

# Antimony String (Model)

ODE model written in Antimony.

In [2]:
# ANTIMONY MODEL
antimony_model = """
model Ecoli_Cfreundii_AmpC_Validation()
 
    // =====================================================================
    // 0. INITIAL CONDITIONS  (all pools start empty; burn-in fills them)
    // =====================================================================
    UDP_NAG = 0.0; UDP_NAG_ep = 0.0; UDP_NAM = 0.0; UDP_NAM_Ala = 0.0
    UDP_NAM_Ala_Glu = 0.0; UDP_NAM_tri_DAP = 0.0; UDP_NAM_penta = 0.0
    Lipid_I_cyto = 0.0; Lipid_II_cyto = 0.0; Lipid_II_peri = 0.0
    PG_UP = 0.0; PG_UT = 0.0; PG_XL = 0.0
    Disacch_P_peri = 0.0; Disacch_T_peri = 0.0
    Disacch_P_cyto = 0.0; Disacch_T_cyto = 0.0; Disacch_Tri = 0.0; Disacch = 0.0
    Anh_P = 0.0; Anh_T = 0.0; Anh_Tri = 0.0
    anhNAM = 0.0; NAM_6_P = 0.0; NAG_6_P = 0.0
    GlcN_6_P = 0.0; GlcN_1_P = 0.0; NAG = 0.0
    Stem_Penta = 0.0; Stem_Tetra = 0.0; Stem_Tri = 0.0
    mRNA_AmpR = 0.0; AmpR = 0.0; AmpR_total = 0.0; mRNA_AmpC = 0.0; AmpC = 0.0
    FtsZ = 0.0; BL = 0.0
    Und_PP_peri = 0.0; Und_P_peri = 0.0; Und_P_cyto = 0.0
    PBP1a = 0.0; PBP1b = 0.0; PBP2 = 0.0; PBP3 = 0.0
    PBP4 = 0.0; PBP5 = 0.0; PBP6 = 0.0; PBP7 = 0.0
    
 
    // =====================================================================
    // 1. GLOBAL / CELL GEOMETRY
    // =====================================================================
    mu = 5.78e-04                  // E. coli LB, ~20 min doubling
    V_cyto = 0.8; V_peri = 0.2
    v_in_NAG   = 6.00e-03
    v_de_novo  = 2.03814e-03
 
    // =====================================================================
    // 2. MUR PATHWAY (all E. coli)
    // =====================================================================
    V_max_MurA = 4.19e-03; K_m_MurA = 0.015; K_inhibit = 0.0009; n_MurA = 1
    V_max_MurB = 3.80e-03; K_m_MurB = 0.015
    V_max_MurC = 8.01e-03; K_m_MurC = 0.04
    V_max_MurD = 7.10e-03; K_m_MurD = 0.0073
    V_max_MurE = 2.86e-03; K_m_MurE = 0.076
    V_max_MurF = 5.47e-03; K_m_MurF = 0.078; K_I_MurF = 0.008 // 1 mM originally
 
    V_max_MraY = 9.23360e-04; K_M_UDP_NAM_pentapeptide = 0.0056; K_M_UndP = 2.00000e-02
    V_max_MurG = 6.68858e-04; K_m_LipidI = 0.02; K_m_UDP_NAG = 0.058
    V_max_MurJ = 2.22e-03; K_M_MurJ = 0.002
 
    // =====================================================================
    // 3. PERIPLASMIC SYNTHASES
    // =====================================================================
    V_max_RodA = 1.03920e-03; K_M_RodA = 0.002
    V_max_FtsW = 1.69e-03; K_M_FtsW = 0.002
 
    k_cat_PBP1a_GT = 5.00000e-01;  K_M_PBP1a_GT = 0.002
    k_cat_PBP1b_GT = 100;  K_M_PBP1b_GT = 0.002
    k_cat_PBP1a_TP = 3.86; K_M_PBP1a_TP = 0.5 // k_cat_PBP1a_TP = 3.86
    k_cat_PBP1b_TP = 3.86; K_M_PBP1b_TP = 0.5
    k_cat_PBP2 = 3.86; K_M_PBP2 = 0.5 // k_cat_PBP2 = 3.86
    k_cat_PBP3 = 3.86; K_M_PBP3 = 0.5
    K_M_acc = 0.5                              // acceptor saturation
 
    // LMW PBPs: carboxypeptidase / endopeptidase
    k_cat_PBP4_CP = 3.47; K_M_PBP4_CP = 0.0204
    k_cat_PBP4_EP = 3.47; K_M_PBP4_EP = 0.0204
    k_cat_PBP5 = 6.56750e-01; K_M_PBP5 = 0.0204
    k_cat_PBP6 = 6.06594e-02; K_M_PBP6 = 0.0204
    k_cat_PBP7 = 3.47; K_M_PBP7 = 0.0204
 
    // =====================================================================
    // 4. HYDROLASES  (Slt = lumped LT; V_max from 40-45%/generation turnover)
    // =====================================================================
    V_max_Slt = 5.00000e-02; K_M_Slt = 1.8
    V_max_Ami_UP = 3.00e-04; K_M_Ami_UP = 1.5
    V_max_Ami_UT = 3.00e-04; K_M_Ami_UT = 1.5
    V_max_Ami_XL = 3.00e-04; K_M_Ami_XL = 1.5
 
    // =====================================================================
    // 5. TRANSPORT AND CYTOPLASMIC RECYCLING
    // =====================================================================
    V_max_AmpG = 1.98e-03; K_M_AmpG = 0.01
 
    V_max_NagZ_Disacch_P   = 1.22e-03; K_M_NagZ_Disacch_P   = 0.104
    V_max_NagZ_Disacch_T   = 1.04e-03; K_M_NagZ_Disacch_T   = 0.095
    V_max_NagZ_Disacch_Tri = 8.61e-04; K_M_NagZ_Disacch_Tri = 0.086
    V_max_NagZ_Disacch     = 3.15e-04; K_M_NagZ_Disacch     = 0.059
 
    V_max_LdcA           = 3.10e-03; K_M_LdcA           = 0.17   // on Anh_T
    V_max_LdcA_Disacch_T = 1.89e-03; K_M_LdcA_Disacch_T = 0.17
    V_max_LdcA_stem      = 1.89e-03; K_M_LdcA_stem      = 0.17   // on free Stem_Tetra
 
    V_max_AmpD_P   = 4.64e-04; K_M_AmpD_P   = 0.5
    V_max_AmpD_T   = 7.59e-04; K_M_AmpD_T   = 0.43
    V_max_AmpD_Tri = 1.05e-03; K_M_AmpD_Tri = 0.36
    V_max_AmpD_Disacch_P   = 1.69e-05; K_M_AmpD_Disacch_P   = 1.76
    V_max_AmpD_Disacch_T   = 2.76e-05; K_M_AmpD_Disacch_T   = 1.514
    V_max_AmpD_Disacch_Tri = 3.83e-05; K_M_AmpD_Disacch_Tri = 1.267
 
    V_max_AnmK = 7.82e-03; K_M_AnmK = 0.5
 
    // Mpl shortcut (E. coli, sourced)
    V_max_Mpl = 3.49e-03; K_M_Mpl_NAM = 0.25
    K_M_Mpl_Tri = 0.1; K_M_Mpl_Penta = 0.1
 
    // Amino-sugar re-entry (E. coli MurQ route; AmgK/MurU/MupP absent)
    V_max_MurQ = 1.68e-03; K_M_MurQ = 0.2
    V_max_NagA = 1.66e-02; K_M_NagA = 0.5
    V_max_NagB = 1.99e-01; K_M_NagB = 0.5
    V_max_NagK = 1.32e-02; K_M_NagK = 0.127
    V_max_GlmS = 5.31e-02; K_M_GlmS_F6P = 2.0; K_M_GlmS_Gln = 0.4
    V_max_GlmM_fwd = 6.21e-02; K_M_GlmM_6P = 0.2
    V_max_GlmM_rev = 6.21e-02; K_M_GlmM_1P = 0.2
    V_max_GlmU = 1.69e-02; K_M_GlmU = 0.1
 
    // Fixed buffered boundary conditions
    Fru_6_P_basal = 0.6
    Gln_basal = 4.0
 
    // =====================================================================
    // 6. UND-P RECYCLING
    // =====================================================================
    V_max_BacA = 4.12609e-03; K_M_BacA = 0.53
    V_max_DedA = 1.25238e-02; K_M_DedA = 0.002
 
    // =====================================================================
    // 7. AmpR / AmpC  (C. freundii genes on pNU305)
    // =====================================================================
    N_copy = 17
    K_R = 0.2; K_A1 = 0.32; K_A2 = 3.2; n_AmpR = 4
    k_tx_rep = 1.17e-06; k_tx_basal = 2.93e-05; k_tx_act = 1.17e-03
    v_tx_AmpR = 6.25e-10; K_autoR = 3.32e-04; n_autoR = 2
    k_tl_AmpR = 0.16;  k_deg_mRNA_AmpR = 3.85e-03; k_deg_p_AmpR = 1.00e-06
    k_tl = 0.16;       k_deg_mRNA_AmpC = 3.85e-03; k_deg_p_AmpC = 1.00e-06
 
    // =====================================================================
    // 8. FtsZ (ftsZ84 temperature-sensitive allele)
    // =====================================================================
    v_synth_FtsZ = 7.04e-06; k_deg_p_FtsZ = 2.70e-04
    K_Z = 0.001; n_Z = 4
 
    // =====================================================================
    // 9. CEFOXITIN
    // =====================================================================
    k_diff_BL = 8.16                  // WEAK estimate - sweep this
    k_cat_AmpC = 0.23; K_M_AmpC = 7.0e-05
    V_max_eff = 0.0; K_M_eff = 0.001  // cefoxitin not an AcrAB substrate
 
    k_acyl_PBP1a = 0.183
    k_acyl_PBP1b = 2.35
    k_acyl_PBP2  = 0.00016
    k_acyl_PBP3  = 0.274
    k_acyl_PBP4  = 16.5
    k_acyl_PBP5  = 2.35
    k_acyl_PBP6  = 2.35
    k_acyl_PBP7  = 16.5
 
    v_synth_PBP1a = 8.90e-08
    v_synth_PBP1b = 4.89e-07
    v_synth_PBP2  = 4.96e-08
    v_synth_PBP3  = 9.36e-08
    v_synth_PBP4  = 5.61e-08
    v_synth_PBP5  = 3.84e-06
    v_synth_PBP6  = 5.24e-06
    v_synth_PBP7  = 7.40e-08
 
    PBP2_ss = 8.59e-05
    PBP3_ss = 1.62e-04

    f_PBP1b_septal = 0.05
 
    // =====================================================================
    // 10. EXPERIMENTAL CONTROLS
    // =====================================================================
    ts_allele = 0     // 0 = WT ftsZ+ ; 1 = ftsZ84
    drug_on   = 0     // 0 = no cefoxitin ; 1 = cefoxitin
    dose      = 0.0187
    t_shift   = 0     // 30->42 C temperature shift
    t_drug    = 1800  // cefoxitin added 30 min AFTER shift
 
    // =====================================================================
    // 11. ALGEBRAIC RULES
    // =====================================================================
    Theta  := 1 - ts_allele * piecewise(1, time >= t_shift, 0)
    BL_ext := drug_on * dose * piecewise(1, time >= t_drug, 0)
 
    Z := Theta * (FtsZ^n_Z) / (K_Z^n_Z + FtsZ^n_Z)
 
    A_acc := (PG_UP + PG_UT) / (K_M_acc + PG_UP + PG_UT)

    XL_f := PG_XL / (PG_UP + PG_UT + PG_XL)
 
    J_TP := A_acc * ( (k_cat_PBP2*PBP2*PG_UP)/(K_M_PBP2 + PG_UP) + Z*(k_cat_PBP3*PBP3*PG_UP)/(K_M_PBP3 + PG_UP) + (k_cat_PBP1a_TP*PBP1a*PG_UP)/(K_M_PBP1a_TP + PG_UP) + Z*(k_cat_PBP1b_TP*PBP1b*PG_UP)/(K_M_PBP1b_TP + PG_UP) )
 
    J_endo := (k_cat_PBP4_EP*PBP4*PG_XL)/(K_M_PBP4_EP + PG_XL) + (k_cat_PBP7*PBP7*PG_XL)/(K_M_PBP7 + PG_XL)
 
    J_RodA   := (PBP2/PBP2_ss) * (V_max_RodA*Lipid_II_peri)/(K_M_RodA + Lipid_II_peri)
    J_FtsW   := Z * (PBP3/PBP3_ss) * (V_max_FtsW*Lipid_II_peri)/(K_M_FtsW + Lipid_II_peri)
    J_PBP1aG := (k_cat_PBP1a_GT*PBP1a*Lipid_II_peri)/(K_M_PBP1a_GT + Lipid_II_peri)
    J_PBP1bG := Z * (k_cat_PBP1b_GT*PBP1b*Lipid_II_peri)/(K_M_PBP1b_GT + Lipid_II_peri)
    J_poly   := J_RodA + J_FtsW + J_PBP1aG + J_PBP1bG
 
    J_MraY := V_max_MraY * (UDP_NAM_penta/(K_M_UDP_NAM_pentapeptide + UDP_NAM_penta)) * (Und_P_cyto/(K_M_UndP + Und_P_cyto))
    J_MurG := (V_max_MurG*Lipid_I_cyto*UDP_NAG)/((K_m_LipidI + Lipid_I_cyto)*(K_m_UDP_NAG + UDP_NAG))
    J_MurA := (V_max_MurA*UDP_NAG)/(K_m_MurA + UDP_NAG) * (1/(1 + (UDP_NAM/K_inhibit)^n_MurA))
    J_MurF := (V_max_MurF*UDP_NAM_tri_DAP)/(K_m_MurF + UDP_NAM_tri_DAP + (UDP_NAM_tri_DAP^2)/K_I_MurF)
    J_MurJ := (V_max_MurJ*Lipid_II_cyto)/(K_M_MurJ + Lipid_II_cyto)
 
    J_Mpl_Tri   := V_max_Mpl * (UDP_NAM/(K_M_Mpl_NAM + UDP_NAM)) * (Stem_Tri/(K_M_Mpl_Tri + Stem_Tri))
    J_Mpl_Penta := V_max_Mpl * (UDP_NAM/(K_M_Mpl_NAM + UDP_NAM)) * (Stem_Penta/(K_M_Mpl_Penta + Stem_Penta))
 
    J_GlmS := V_max_GlmS * (Fru_6_P_basal/(K_M_GlmS_F6P + Fru_6_P_basal)) * (Gln_basal/(K_M_GlmS_Gln + Gln_basal))
    J_GlmM := (V_max_GlmM_fwd*(GlcN_6_P/K_M_GlmM_6P) - V_max_GlmM_rev*(GlcN_1_P/K_M_GlmM_1P)) / (1 + GlcN_6_P/K_M_GlmM_6P + GlcN_1_P/K_M_GlmM_1P)
 
    J_Slt_UP := (V_max_Slt*PG_UP)/(K_M_Slt + PG_UP)
    J_Slt_UT := (V_max_Slt*PG_UT)/(K_M_Slt + PG_UT)
    J_AmpG_P := (V_max_AmpG*Disacch_P_peri)/(K_M_AmpG + Disacch_P_peri)
    J_AmpG_T := (V_max_AmpG*Disacch_T_peri)/(K_M_AmpG + Disacch_T_peri)
 
    J_NagZ_P   := (V_max_NagZ_Disacch_P*Disacch_P_cyto)/(K_M_NagZ_Disacch_P + Disacch_P_cyto)
    J_NagZ_T   := (V_max_NagZ_Disacch_T*Disacch_T_cyto)/(K_M_NagZ_Disacch_T + Disacch_T_cyto)
    J_NagZ_Tri := (V_max_NagZ_Disacch_Tri*Disacch_Tri)/(K_M_NagZ_Disacch_Tri + Disacch_Tri)
    J_NagZ_D   := (V_max_NagZ_Disacch*Disacch)/(K_M_NagZ_Disacch + Disacch)
 
    J_AmpD_P   := (V_max_AmpD_P*Anh_P)/(K_M_AmpD_P + Anh_P)
    J_AmpD_T   := (V_max_AmpD_T*Anh_T)/(K_M_AmpD_T + Anh_T)
    J_AmpD_Tri := (V_max_AmpD_Tri*Anh_Tri)/(K_M_AmpD_Tri + Anh_Tri)
    J_AmpD_DP  := (V_max_AmpD_Disacch_P*Disacch_P_cyto)/(K_M_AmpD_Disacch_P + Disacch_P_cyto)
    J_AmpD_DT  := (V_max_AmpD_Disacch_T*Disacch_T_cyto)/(K_M_AmpD_Disacch_T + Disacch_T_cyto)
    J_AmpD_DTri := (V_max_AmpD_Disacch_Tri*Disacch_Tri)/(K_M_AmpD_Disacch_Tri + Disacch_Tri)
 
    J_LdcA_AnhT := (V_max_LdcA*Anh_T)/(K_M_LdcA + Anh_T)
    J_LdcA_DT   := (V_max_LdcA_Disacch_T*Disacch_T_cyto)/(K_M_LdcA_Disacch_T + Disacch_T_cyto)
    J_LdcA_stem := (V_max_LdcA_stem*Stem_Tetra)/(K_M_LdcA_stem + Stem_Tetra)
 
    J_BacA := (V_max_BacA*Und_PP_peri)/(K_M_BacA + Und_PP_peri)
    J_DedA := (V_max_DedA*Und_P_peri)/(K_M_DedA + Und_P_peri)

        // --- newly named terms, so every equation's pieces are individually inspectable ---
    J_MurB := (V_max_MurB*UDP_NAG_ep)/(K_m_MurB + UDP_NAG_ep)
    J_MurC := (V_max_MurC*UDP_NAM)/(K_m_MurC + UDP_NAM)
    J_MurD := (V_max_MurD*UDP_NAM_Ala)/(K_m_MurD + UDP_NAM_Ala)
    J_MurE := (V_max_MurE*UDP_NAM_Ala_Glu)/(K_m_MurE + UDP_NAM_Ala_Glu)
    J_GlmU := (V_max_GlmU*GlcN_1_P)/(K_M_GlmU + GlcN_1_P)

    J_PBP5_CP := (k_cat_PBP5*PBP5*PG_UP)/(K_M_PBP5 + PG_UP)
    J_PBP6_CP := (k_cat_PBP6*PBP6*PG_UP)/(K_M_PBP6 + PG_UP)
    J_PBP4_CP := (k_cat_PBP4_CP*PBP4*PG_UP)/(K_M_PBP4_CP + PG_UP)
    J_Ami_UP  := (V_max_Ami_UP*PG_UP)/(K_M_Ami_UP + PG_UP)
    J_Ami_UT  := (V_max_Ami_UT*PG_UT)/(K_M_Ami_UT + PG_UT)
    J_Ami_XL  := (V_max_Ami_XL*PG_XL)/(K_M_Ami_XL + PG_XL)

    J_AnmK := (V_max_AnmK*anhNAM)/(K_M_AnmK + anhNAM)
    J_MurQ := (V_max_MurQ*NAM_6_P)/(K_M_MurQ + NAM_6_P)
    J_NagK := (V_max_NagK*NAG)/(K_M_NagK + NAG)
    J_NagA := (V_max_NagA*NAG_6_P)/(K_M_NagA + NAG_6_P)
    J_NagB := (V_max_NagB*GlcN_6_P)/(K_M_NagB + GlcN_6_P)

    J_tx_AmpR := N_copy*v_tx_AmpR*(1/(1 + (AmpR_total/K_autoR)^n_autoR))
    J_tl_AmpR := k_tl_AmpR*mRNA_AmpR

    J_tx_AmpC_rep   := N_copy*k_tx_rep*(UDP_NAM_penta/K_R)^n_AmpR*AmpR
    J_tx_AmpC_basal := N_copy*k_tx_basal*AmpR
    J_tx_AmpC_act   := N_copy*k_tx_act*((Anh_P/K_A1)^n_AmpR + (Disacch_P_cyto/K_A2)^n_AmpR)*AmpR
    J_tl_AmpC := (V_cyto/V_peri)*k_tl*mRNA_AmpC

    J_diff_BL := k_diff_BL*(BL_ext - BL)
    J_AmpC_hydrolysis := (k_cat_AmpC*AmpC*BL)/(K_M_AmpC + BL)
    J_eff := (V_max_eff*BL)/(K_M_eff + BL)
    J_acyl_PBP1a := k_acyl_PBP1a*PBP1a*BL
    J_acyl_PBP1b := k_acyl_PBP1b*PBP1b*BL
    J_acyl_PBP2  := k_acyl_PBP2*PBP2*BL
    J_acyl_PBP3  := k_acyl_PBP3*PBP3*BL
    J_acyl_PBP4  := k_acyl_PBP4*PBP4*BL
    J_acyl_PBP5  := k_acyl_PBP5*PBP5*BL
    J_acyl_PBP6  := k_acyl_PBP6*PBP6*BL
    J_acyl_PBP7  := k_acyl_PBP7*PBP7*BL

    XL_f := PG_XL / (PG_UP + PG_UT + PG_XL)

    denom_AmpR := 1 + (UDP_NAM_penta/K_R)^n_AmpR + (Anh_P/K_A1)^n_AmpR + (Disacch_P_cyto/K_A2)^n_AmpR
    AmpR := AmpR_total / denom_AmpR
 
    // =====================================================================
    // 12. CYTOPLASMIC PRECURSOR BIOSYNTHESIS
    // =====================================================================
    UDP_NAG' = v_in_NAG + (V_max_GlmU*GlcN_1_P)/(K_M_GlmU + GlcN_1_P) - J_MurA - J_MurG - mu*UDP_NAG
    UDP_NAG_ep' = J_MurA - (V_max_MurB*UDP_NAG_ep)/(K_m_MurB + UDP_NAG_ep) - mu*UDP_NAG_ep
    UDP_NAM' = (V_max_MurB*UDP_NAG_ep)/(K_m_MurB + UDP_NAG_ep) - (V_max_MurC*UDP_NAM)/(K_m_MurC + UDP_NAM) - J_Mpl_Penta - J_Mpl_Tri - mu*UDP_NAM
    UDP_NAM_Ala' = (V_max_MurC*UDP_NAM)/(K_m_MurC + UDP_NAM) - (V_max_MurD*UDP_NAM_Ala)/(K_m_MurD + UDP_NAM_Ala) - mu*UDP_NAM_Ala
    UDP_NAM_Ala_Glu' = (V_max_MurD*UDP_NAM_Ala)/(K_m_MurD + UDP_NAM_Ala) - (V_max_MurE*UDP_NAM_Ala_Glu)/(K_m_MurE + UDP_NAM_Ala_Glu) - mu*UDP_NAM_Ala_Glu
    UDP_NAM_tri_DAP' = (V_max_MurE*UDP_NAM_Ala_Glu)/(K_m_MurE + UDP_NAM_Ala_Glu) + J_Mpl_Tri - J_MurF - mu*UDP_NAM_tri_DAP
    UDP_NAM_penta' = J_MurF + J_Mpl_Penta - J_MraY - mu*UDP_NAM_penta
    Lipid_I_cyto' = J_MraY - J_MurG - mu*Lipid_I_cyto
    Lipid_II_cyto' = J_MurG - J_MurJ - mu*Lipid_II_cyto
 
    // =====================================================================
    // 13. PERIPLASMIC REACTIONS
    // =====================================================================
    Lipid_II_peri' = (V_cyto/V_peri)*J_MurJ - J_poly - mu*Lipid_II_peri
 
    PG_UP' = J_poly - J_TP - (k_cat_PBP5*PBP5*PG_UP)/(K_M_PBP5 + PG_UP) - (k_cat_PBP6*PBP6*PG_UP)/(K_M_PBP6 + PG_UP) - (k_cat_PBP4_CP*PBP4*PG_UP)/(K_M_PBP4_CP + PG_UP) - J_Slt_UP - (V_max_Ami_UP*PG_UP)/(K_M_Ami_UP + PG_UP) - mu*PG_UP
 
    PG_UT' = (k_cat_PBP5*PBP5*PG_UP)/(K_M_PBP5 + PG_UP) + (k_cat_PBP6*PBP6*PG_UP)/(K_M_PBP6 + PG_UP) + (k_cat_PBP4_CP*PBP4*PG_UP)/(K_M_PBP4_CP + PG_UP) + J_endo - J_Slt_UT - (V_max_Ami_UT*PG_UT)/(K_M_Ami_UT + PG_UT) - mu*PG_UT
 
    PG_XL' = J_TP - J_endo - (V_max_Ami_XL*PG_XL)/(K_M_Ami_XL + PG_XL) - mu*PG_XL
 
    Disacch_P_peri' = J_Slt_UP - J_AmpG_P - mu*Disacch_P_peri
    Disacch_T_peri' = J_Slt_UT - J_AmpG_T - mu*Disacch_T_peri
 
    // =====================================================================
    // 14. CYTOPLASMIC RECYCLING
    // =====================================================================
    Disacch_P_cyto' = (V_peri/V_cyto)*J_AmpG_P - J_NagZ_P - J_AmpD_DP - mu*Disacch_P_cyto
    Disacch_T_cyto' = (V_peri/V_cyto)*J_AmpG_T - J_NagZ_T - J_AmpD_DT - J_LdcA_DT - mu*Disacch_T_cyto
    Disacch_Tri' = J_LdcA_DT - J_NagZ_Tri - J_AmpD_DTri - mu*Disacch_Tri
    Disacch' = J_AmpD_DP + J_AmpD_DT + J_AmpD_DTri - J_NagZ_D - mu*Disacch
 
    Anh_P' = J_NagZ_P - J_AmpD_P - mu*Anh_P
    Anh_T' = J_NagZ_T - J_AmpD_T - J_LdcA_AnhT - mu*Anh_T
    Anh_Tri' = J_NagZ_Tri + J_LdcA_AnhT - J_AmpD_Tri - mu*Anh_Tri
 
    anhNAM' = J_AmpD_P + J_AmpD_T + J_AmpD_Tri + J_NagZ_D - (V_max_AnmK*anhNAM)/(K_M_AnmK + anhNAM) - mu*anhNAM
    NAM_6_P' = (V_max_AnmK*anhNAM)/(K_M_AnmK + anhNAM) - (V_max_MurQ*NAM_6_P)/(K_M_MurQ + NAM_6_P) - mu*NAM_6_P
    NAG_6_P' = (V_max_MurQ*NAM_6_P)/(K_M_MurQ + NAM_6_P) + (V_max_NagK*NAG)/(K_M_NagK + NAG) - (V_max_NagA*NAG_6_P)/(K_M_NagA + NAG_6_P) - mu*NAG_6_P
    GlcN_6_P' = (V_max_NagA*NAG_6_P)/(K_M_NagA + NAG_6_P) + J_GlmS - (V_max_NagB*GlcN_6_P)/(K_M_NagB + GlcN_6_P) - J_GlmM - mu*GlcN_6_P
    GlcN_1_P' = J_GlmM - (V_max_GlmU*GlcN_1_P)/(K_M_GlmU + GlcN_1_P) - mu*GlcN_1_P
    NAG' = J_NagZ_D + J_NagZ_P + J_NagZ_T + J_NagZ_Tri - (V_max_NagK*NAG)/(K_M_NagK + NAG) - mu*NAG
 
    Stem_Penta' = J_AmpD_DP + J_AmpD_P - J_Mpl_Penta - mu*Stem_Penta
    Stem_Tetra' = J_AmpD_DT + J_AmpD_T - J_LdcA_stem - mu*Stem_Tetra
    Stem_Tri' = J_AmpD_DTri + J_AmpD_Tri + J_LdcA_stem - J_Mpl_Tri - mu*Stem_Tri
 
    // =====================================================================
    // 15. AmpR / AmpC EXPRESSION
    // =====================================================================
    mRNA_AmpR' = N_copy*v_tx_AmpR*(1/(1 + (AmpR_total/K_autoR)^n_autoR)) - (k_deg_mRNA_AmpR + mu)*mRNA_AmpR
    AmpR_total' = k_tl_AmpR*mRNA_AmpR - (k_deg_p_AmpR + mu)*AmpR_total
 
    mRNA_AmpC' = N_copy*( k_tx_rep*(UDP_NAM_penta/K_R)^n_AmpR + k_tx_basal + k_tx_act*( (Anh_P/K_A1)^n_AmpR + (Disacch_P_cyto/K_A2)^n_AmpR ) )*AmpR - (k_deg_mRNA_AmpC + mu)*mRNA_AmpC
    AmpC' = (V_cyto/V_peri)*k_tl*mRNA_AmpC - (k_deg_p_AmpC + mu)*AmpC
 
    // =====================================================================
    // 16. FtsZ
    // =====================================================================
    FtsZ' = v_synth_FtsZ - (k_deg_p_FtsZ + mu)*FtsZ
 
    // =====================================================================
    // 17. CEFOXITIN AND PBP ACYLATION
    // =====================================================================
    BL' = k_diff_BL*(BL_ext - BL) - (k_cat_AmpC*AmpC*BL)/(K_M_AmpC + BL) - (V_max_eff*BL)/(K_M_eff + BL) - (k_acyl_PBP1a*PBP1a + k_acyl_PBP1b*PBP1b + k_acyl_PBP2*PBP2 + k_acyl_PBP3*PBP3 + k_acyl_PBP4*PBP4 + k_acyl_PBP5*PBP5 + k_acyl_PBP6*PBP6 + k_acyl_PBP7*PBP7)*BL - mu*BL
 
    PBP1a' = v_synth_PBP1a - k_acyl_PBP1a*PBP1a*BL - mu*PBP1a
    PBP1b' = v_synth_PBP1b - k_acyl_PBP1b*PBP1b*BL - mu*PBP1b
    PBP2'  = v_synth_PBP2  - k_acyl_PBP2*PBP2*BL   - mu*PBP2
    PBP3'  = v_synth_PBP3  - k_acyl_PBP3*PBP3*BL   - mu*PBP3
    PBP4'  = v_synth_PBP4  - k_acyl_PBP4*PBP4*BL   - mu*PBP4
    PBP5'  = v_synth_PBP5  - k_acyl_PBP5*PBP5*BL   - mu*PBP5
    PBP6'  = v_synth_PBP6  - k_acyl_PBP6*PBP6*BL   - mu*PBP6
    PBP7'  = v_synth_PBP7  - k_acyl_PBP7*PBP7*BL   - mu*PBP7
 
    // =====================================================================
    // 18. UND-P RECYCLING
    // =====================================================================
    Und_PP_peri' = v_de_novo + J_poly - J_BacA - mu*Und_PP_peri
    Und_P_peri'  = J_BacA - J_DedA - mu*Und_P_peri
    Und_P_cyto'  = (V_peri/V_cyto)*J_DedA - J_MraY - mu*Und_P_cyto
 
end
""" 

# Steady-State Helper Functions

## Solver Parameters

In [3]:
STEADY_STATE_RESIDUAL_TOL = 1e-6
FALLBACK_MAX_STEPS        = 200000
FALLBACK_BURNIN_TIME      = 500000
ABS_TOL_DYDT              = 1e-10

## Residual Finder

This function assesses whether the system has reached a **steady state** by testing if every chemical's rate of change meets absolute or relative stability thresholds, set before. It is used as a helper function inside `reach_steady_state()`.

In [4]:
def verify_residual(r, rel_tol=STEADY_STATE_RESIDUAL_TOL, abs_tol=ABS_TOL_DYDT):
    rates_arr = r.getRatesOfChangeNamedArray() # Gets rates of change of each species
    species_ids = list(rates_arr.colnames) # Gets the species list
    dydt = np.array(rates_arr).flatten() # Array of rates of change of each species
    y = np.array([r[sid] for sid in species_ids]) # Array of concentration of each species
 
    abs_ok = np.abs(dydt) < abs_tol # Absolute Check
    rel_ratio = np.abs(dydt) / (np.abs(y) + 1e-12) # Relative Ratio Calculation
    rel_ok = rel_ratio < rel_tol # Relative Check
    converged = abs_ok | rel_ok # A species is considered at steady-state if it passes the absolute OR the relative check
 
    if np.all(converged): # If every single chemical in the entire model passed the test, the system is at steady state
        return 0.0
    return float(np.max(rel_ratio[~converged])) # If even one chemical failed, system is NOT at steady-state. Returns the chemical with the highest relative error.

## Steady-State Finder

The function encoded below finds the steady-state of each species in the model by using a three-phase approach. 
1. **Initial Burn-In (Phase 1)** - It runs a long (500,000 s) time-course simulation (using CVODE) to bring the system close to equilibrium.
2. **Newton Method (Phase 2)** - After the burn-in phase, it uses an algebraic root-finder (Newton's method) to instantly and precisely calculate the exact steady-state concentrations.
3. **Brute-Force Fallback (Phase 3)** - If the Newton method fails, it reverts to the time-stepper and simulates forward for another massive block of time (500,000 s), and we assume the concentrations at the end of the long run (totalling 1 million seconds) to be in steady-state.

*Note*: To check whether steady-state is reached, we use the `verify_residual()` function to confirm.

In [5]:
def reach_steady_state(r, warm_start_time=FALLBACK_BURNIN_TIME):
    # Setting the solver parameters
    integrator = r.getIntegrator()
    integrator.setValue('relative_tolerance', 1e-10)
    integrator.setValue('absolute_tolerance', 1e-12)
    integrator.maximum_num_steps = FALLBACK_MAX_STEPS

    # Initial Burn-in (Phase 1)
    try:
        r.simulate(0, warm_start_time, 2)
    except RuntimeError: # If solver throws an error here, terminate the function with a failure message
        return False, 'integration_fallback_exception', np.nan

    # Newton Method (Phase 2)
    try:
        ss_solver = r.getSteadyStateSolver()
        ss_solver.setValue('allow_approx', True)
        ss_solver.setValue('relative_tol', 1e-12)
        r.steadyState()
        residual = verify_residual(r) # Get residual
        if np.isfinite(residual) and residual < STEADY_STATE_RESIDUAL_TOL: # If residual is below set threshold, assume success (steady-state reached) and terminate the function
            return True, 'newton_warm', residual
    except RuntimeError:
        pass

    # Integration Fallback (Phase 3)
    try:
        r.simulate(0, FALLBACK_BURNIN_TIME, 2)
        residual = verify_residual(r) # Check residual again
        if np.isfinite(residual) and residual < STEADY_STATE_RESIDUAL_TOL: # If residual is below accepcted threshold, assume success and terminate function
            return True, 'integration_fallback', residual
        else:
            return False, 'no_steady_state', residual # If residual is large, inform steady-state has not been reached through the integration fallback
    except RuntimeError: # If solver throws an error here, integration fallback has failed to simulate for the set time
        return False, 'integration_fallback_exception', np.nan

# Simulation Helper Functions

## Literature Targets

The data points from *30 min* after cefoxitin addition are taken from **Uehara & Park 2002, Table 2, rows 1 and 3**. 


The *60* and *90 min* data points are obtained from **Figure 1** of the same paper.

In [6]:
# Absolute Targets

LIT_TARGETS_ABS = {
    "LIT_NONINDUCED_WT_30": 0.5,
    "LIT_INDUCED_WT_30": 3.1,
    "LIT_NONINDUCED_FTSZ_30": 0.5,
    "LIT_INDUCED_FTSZ_30": 0.7,
    "LIT_NONINDUCED_WT_60": 0.4,
    "LIT_INDUCED_WT_60": 2.35,
    "LIT_NONINDUCED_FTSZ_60": 0.6,
    "LIT_INDUCED_FTSZ_60": 0.85,
    "LIT_NONINDUCED_WT_90": 0.5,
    "LIT_INDUCED_WT_90": 1.65,
    "LIT_NONINDUCED_FTSZ_90": 0.55,
    "LIT_INDUCED_FTSZ_90": 0.55
}

# Fold Targets

## 30 min
FOLD_WT_TARGET_30 = LIT_TARGETS_ABS["LIT_INDUCED_WT_30"] / LIT_TARGETS_ABS["LIT_NONINDUCED_WT_30"]          # 6.2
FOLD_FTSZ_TARGET_30 = LIT_TARGETS_ABS["LIT_INDUCED_FTSZ_30"] / LIT_TARGETS_ABS["LIT_NONINDUCED_FTSZ_30"]    # 1.4


## 60 min
FOLD_WT_TARGET_60 = LIT_TARGETS_ABS["LIT_INDUCED_WT_60"] / LIT_TARGETS_ABS["LIT_NONINDUCED_WT_60"]          # 5.8
FOLD_FTSZ_TARGET_60 = LIT_TARGETS_ABS["LIT_INDUCED_FTSZ_60"] / LIT_TARGETS_ABS["LIT_NONINDUCED_FTSZ_60"]    # 1.4

## 90 min
FOLD_WT_TARGET_90 = LIT_TARGETS_ABS["LIT_INDUCED_WT_90"] / LIT_TARGETS_ABS["LIT_NONINDUCED_WT_90"]          # 3.3
FOLD_FTSZ_TARGET_90 = LIT_TARGETS_ABS["LIT_INDUCED_FTSZ_90"] / LIT_TARGETS_ABS["LIT_NONINDUCED_FTSZ_90"]    # 1

## Fold change dictionary
LIT_TARGETS_FOLD = {
    "FOLD_WT_TARGET_30": FOLD_WT_TARGET_30,
    "FOLD_FTSZ_TARGET_30": FOLD_FTSZ_TARGET_30,
    "FOLD_WT_TARGET_60": FOLD_WT_TARGET_60,
    "FOLD_FTSZ_TARGET_60": FOLD_FTSZ_TARGET_60,
    "FOLD_WT_TARGET_90": FOLD_WT_TARGET_90,
    "FOLD_FTSZ_TARGET_90": FOLD_FTSZ_TARGET_90
}

# Rho Targets

## Calcultating rho values for 30 min, 60 min, and 90 min
RHO_TARGET_30 = FOLD_FTSZ_TARGET_30 / FOLD_WT_TARGET_30               # ~0.23
RHO_TARGET_60 = FOLD_FTSZ_TARGET_60 / FOLD_WT_TARGET_60               # ~0.24
RHO_TARGET_90 = FOLD_FTSZ_TARGET_90 / FOLD_WT_TARGET_90               # ~0.30

## Building the dictionary
RHO_TARGETS = {
    'RHO_TARGET_30': RHO_TARGET_30,
    'RHO_TARGET_60': RHO_TARGET_60,
    'RHO_TARGET_90': RHO_TARGET_90
}

## Experiment Controls and Time Points

The timings were directly obtained from **Materials and Methods** of **Uehara and Park (2002)**.
- t=0        : shared culture at 30 C, 15 Klett units (burn-in state)
- t=0..1800  : shift to 42 C (Theta gates ftsZ84 branches here)
- t=1800     : cefoxitin added to the two "+BL" branches
- t=1800..3600: 30 min induction period
- t=3600,5400,7200     : sampling points 

In [7]:
T_SHIFT = 0
T_DRUG = 1800 # Drug added at 30 min from start
T_SAMPLES = [3600, 5400, 7200] # 30 min, 60 min, 90 min (with respect to drug addition)
T_END = 7200 # time-course plotted over the full protocol (120 min)
N_POINTS = 721 # ~10 s resolution for smooth curves

## Simulation Function

This function reproduces the four experimental conditions described by **Uehara and Park (2002)**. It first invokes `reach_steady_state()` to simulate the initial 30$\degree$C incubation period. From this shared state, the function runs four parallel 2-hour simulations. It returns a time-course dataset (721 points) tracking every chemical species, along with the final 2-hour endpoint concentrations for direct comparison against the published literature.

In [8]:
def run_uehara_park_validation():
    r = te.loada(antimony_model) # Loads the Antimony model and creates a RoadRunner simulation object

    r.ts_allele = 0
    r.drug_on = 0
    
    ok, method, residual = reach_steady_state(r) # Reach steady state
    
    if not ok:
        raise RuntimeError(
            f"Shared 30 C burn-in did not converge (method={method}, "
            f"residual={residual}). Fix this before trusting anything below."
        )
        
    print(f"Shared burn-in converged via '{method}', residual={residual:.2e}")
    
    r.saveState('shared_30C_state') # Save the steady state model

    # Defining the 4 experimental conditions
    conditions = {
        '1_WT_noBL':    dict(ts_allele=0, drug_on=0),
        '2_WT_plusBL':  dict(ts_allele=0, drug_on=1),
        '3_ftsZ_noBL':  dict(ts_allele=1, drug_on=0),
        '4_ftsZ_plusBL': dict(ts_allele=1, drug_on=1),
    }

    # Specify tracked variables

    ## Steady-state variables
    all_species = [s for s in r.selections if s != 'time']

    ## Flux terms and other important variables
    extra_assignment_rules = [
    'Z', 'BL_ext', 'Theta', 'mu', 'XL_f', 'AmpR', 'denom_AmpR',
    'J_poly', 'J_TP', 'J_endo', 'J_RodA', 'J_FtsW', 'J_PBP1aG', 'J_PBP1bG',
    'J_MraY', 'J_MurA', 'J_MurB', 'J_MurC', 'J_MurD', 'J_MurE', 'J_MurF',
    'J_MurG', 'J_MurJ', 'J_GlmU', 'J_Mpl_Tri', 'J_Mpl_Penta', 'J_GlmS', 'J_GlmM',
    'J_PBP5_CP', 'J_PBP6_CP', 'J_PBP4_CP', 'J_Ami_UP', 'J_Ami_UT', 'J_Ami_XL',
    'J_Slt_UP', 'J_Slt_UT', 'J_AmpG_P', 'J_AmpG_T',
    'J_NagZ_P', 'J_NagZ_T', 'J_NagZ_Tri', 'J_NagZ_D',
    'J_AmpD_P', 'J_AmpD_T', 'J_AmpD_Tri', 'J_AmpD_DP', 'J_AmpD_DT', 'J_AmpD_DTri',
    'J_LdcA_AnhT', 'J_LdcA_DT', 'J_LdcA_stem', 'J_AnmK', 'J_MurQ', 'J_NagK', 'J_NagA', 'J_NagB',
    'J_BacA', 'J_DedA',
    'J_tx_AmpR', 'J_tl_AmpR', 'J_tx_AmpC_rep', 'J_tx_AmpC_basal', 'J_tx_AmpC_act', 'J_tl_AmpC',
    'J_diff_BL', 'J_AmpC_hydrolysis', 'J_eff',
    'J_acyl_PBP1a', 'J_acyl_PBP1b', 'J_acyl_PBP2', 'J_acyl_PBP3',
    'J_acyl_PBP4', 'J_acyl_PBP5', 'J_acyl_PBP6', 'J_acyl_PBP7',
    'v_in_NAG', 'v_de_novo', 'v_synth_FtsZ',
    'v_synth_PBP1a', 'v_synth_PBP1b', 'v_synth_PBP2', 'v_synth_PBP3',
    'v_synth_PBP4', 'v_synth_PBP5', 'v_synth_PBP6', 'v_synth_PBP7',
    'k_deg_mRNA_AmpR', 'k_deg_p_AmpR', 'k_deg_mRNA_AmpC', 'k_deg_p_AmpC', 'k_deg_p_FtsZ']

    ## Combining time with the tracked variables above to form the final set of selected variables
    selected_vars = ['time'] + all_species + extra_assignment_rules

    # Dictionaries for result storage
    timecourses = {} # Stores each condition's timecourse of selected variables
    sample_points = {} # Stores values of selected variables in each condition at specific sampling time points
    endpoint = {} # Stores endpoint values of selected variables in each condition

    # GOATED LOOP
    for name, ctrl in conditions.items():
        r.loadState('shared_30C_state')
        r.ts_allele = ctrl['ts_allele']
        r.drug_on = ctrl['drug_on']
        r.t_shift = T_SHIFT
        r.t_drug = T_DRUG

        r.getIntegrator().maximum_num_steps = FALLBACK_MAX_STEPS
        
        result = r.simulate(0, T_END, N_POINTS, selected_vars)
        
        timecourses[name] = result

        sample_points[name] = {}

        for timepoint in T_SAMPLES:
            sample_points[name][timepoint] = {v: result[v][int(timepoint / 10)] for v in selected_vars}

        endpoint[name] = {v: result[v][-1] for v in selected_vars}

    return timecourses, sample_points, endpoint

## Metrics Generator

This function computes the final **AmpC fold induction** and the **interaction ratio ($\rho$)** across the four experimental conditions. It structures the calculated metrics and concentrations into summary tables. The model's output directly aligns against the literature targets from **Uehara and Park (2002)** for validation.

In [9]:
def compute_metrics(sample_points):
    EPS = 1e-30 # to avoid division by zero

    # Model Values at 30 min, 60 min, and 90 min
    
    ## 30 min metrics
    AmpC_1_30 = sample_points['1_WT_noBL'][3600]['AmpC']
    AmpC_2_30 = sample_points['2_WT_plusBL'][3600]['AmpC']
    AmpC_3_30 = sample_points['3_ftsZ_noBL'][3600]['AmpC']
    AmpC_4_30 = sample_points['4_ftsZ_plusBL'][3600]['AmpC']
 
    fold_WT_30 = AmpC_2_30 / (AmpC_1_30 + EPS)
    fold_ftsZ_30 = AmpC_4_30 / (AmpC_3_30 + EPS)
    rho_30 = fold_ftsZ_30 / (fold_WT_30 + EPS)

    ## 60 min metrics
    AmpC_1_60 = sample_points['1_WT_noBL'][5400]['AmpC']
    AmpC_2_60 = sample_points['2_WT_plusBL'][5400]['AmpC']
    AmpC_3_60 = sample_points['3_ftsZ_noBL'][5400]['AmpC']
    AmpC_4_60 = sample_points['4_ftsZ_plusBL'][5400]['AmpC']
 
    fold_WT_60 = AmpC_2_60 / (AmpC_1_60 + EPS)
    fold_ftsZ_60 = AmpC_4_60 / (AmpC_3_60 + EPS)
    rho_60 = fold_ftsZ_60 / (fold_WT_60 + EPS)

    ## 90 min metrics
    AmpC_1_90 = sample_points['1_WT_noBL'][7200]['AmpC']
    AmpC_2_90 = sample_points['2_WT_plusBL'][7200]['AmpC']
    AmpC_3_90 = sample_points['3_ftsZ_noBL'][7200]['AmpC']
    AmpC_4_90 = sample_points['4_ftsZ_plusBL'][7200]['AmpC']
 
    fold_WT_90 = AmpC_2_90 / (AmpC_1_90 + EPS)
    fold_ftsZ_90 = AmpC_4_90 / (AmpC_3_90 + EPS)
    rho_90 = fold_ftsZ_90 / (fold_WT_90 + EPS)

    # Building the data frames
    
    ## Absolute values
    summary = pd.DataFrame([
        {
            'Condition': 'WT, noninduced', 
            'Model_AmpC_mM_30': AmpC_1_30,
            'Lit_activity_umol_mg_min_30': LIT_TARGETS_ABS["LIT_NONINDUCED_WT_30"],
            'Model_AmpC_mM_60': AmpC_1_60,
            'Lit_activity_umol_mg_min_60': LIT_TARGETS_ABS["LIT_NONINDUCED_WT_60"],
            'Model_AmpC_mM_90': AmpC_1_90,
            'Lit_activity_umol_mg_min_90': LIT_TARGETS_ABS["LIT_NONINDUCED_WT_90"]
        },
        {
            'Condition': 'WT, induced', 
            'Model_AmpC_mM_30': AmpC_2_30,
            'Lit_activity_umol_mg_min_30': LIT_TARGETS_ABS["LIT_INDUCED_WT_30"],
            'Model_AmpC_mM_60': AmpC_2_60,
            'Lit_activity_umol_mg_min_60': LIT_TARGETS_ABS["LIT_INDUCED_WT_60"],
            'Model_AmpC_mM_90': AmpC_2_90,
            'Lit_activity_umol_mg_min_90': LIT_TARGETS_ABS["LIT_INDUCED_WT_90"]
        },
        {
            'Condition': 'ftsZ84, noninduced', 
            'Model_AmpC_mM_30': AmpC_3_30,
            'Lit_activity_umol_mg_min_30': LIT_TARGETS_ABS["LIT_NONINDUCED_FTSZ_30"],
            'Model_AmpC_mM_60': AmpC_3_60,
            'Lit_activity_umol_mg_min_60': LIT_TARGETS_ABS["LIT_NONINDUCED_FTSZ_60"],
            'Model_AmpC_mM_90': AmpC_3_90,
            'Lit_activity_umol_mg_min_90': LIT_TARGETS_ABS["LIT_NONINDUCED_FTSZ_90"]
        },
        {
            'Condition': 'ftsZ84, induced', 
            'Model_AmpC_mM_30': AmpC_4_30,
            'Lit_activity_umol_mg_min_30': LIT_TARGETS_ABS["LIT_INDUCED_FTSZ_30"],
            'Model_AmpC_mM_60': AmpC_4_60,
            'Lit_activity_umol_mg_min_60': LIT_TARGETS_ABS["LIT_INDUCED_FTSZ_60"],
            'Model_AmpC_mM_90': AmpC_4_90,
            'Lit_activity_umol_mg_min_90': LIT_TARGETS_ABS["LIT_INDUCED_FTSZ_90"]
        }
    ])

    ## Fold changes
    fold_summary = pd.DataFrame([
        {
            'Metric': 'Fold induction, WT', 
            'Model_30': fold_WT_30, 
            'Uehara_Park_2002_30': LIT_TARGETS_FOLD["FOLD_WT_TARGET_30"], 
            'Model_60': fold_WT_60, 
            'Uehara_Park_2002_60': LIT_TARGETS_FOLD["FOLD_WT_TARGET_60"],
            'Model_90': fold_WT_90, 
            'Uehara_Park_2002_90': LIT_TARGETS_FOLD["FOLD_WT_TARGET_90"]
        },
        {
            'Metric': 'Fold induction, ftsZ84', 
            'Model_30': fold_ftsZ_30, 
            'Uehara_Park_2002_30': LIT_TARGETS_FOLD["FOLD_FTSZ_TARGET_30"], 
            'Model_60': fold_ftsZ_60, 
            'Uehara_Park_2002_60': LIT_TARGETS_FOLD["FOLD_FTSZ_TARGET_60"],
            'Model_90': fold_ftsZ_90, 
            'Uehara_Park_2002_90': LIT_TARGETS_FOLD["FOLD_FTSZ_TARGET_90"]
        },
        {
            'Metric': 'rho (interaction ratio)', 
            'Model_30': rho_30, 
            'Uehara_Park_2002_30': RHO_TARGETS['RHO_TARGET_30'],
            'Model_60': rho_60,
            'Uehara_Park_2002_60': RHO_TARGETS['RHO_TARGET_60'],
            'Model_90': rho_90,
            'Uehara_Park_2002_90': RHO_TARGETS['RHO_TARGET_90']
        }
    ])
 
    return summary, fold_summary

# Plotting Helper Functions

## Plotting Parameters

In [10]:
COLORS = {
    '1_WT_noBL': '#888780',
    '2_WT_plusBL': '#185FA5',
    '3_ftsZ_noBL': '#F0997B',
    '4_ftsZ_plusBL': '#D85A30',
}
LABELS = {
    '1_WT_noBL': 'WT, no BL',
    '2_WT_plusBL': 'WT, +BL',
    '3_ftsZ_noBL': 'ftsZ84, no BL',
    '4_ftsZ_plusBL': 'ftsZ84, +BL',
}

## Timecourse Plot Generator

This function generates a separate time-course plot for each selected model variable, comparing its simulated dynamics across the four experimental conditions. Each plot includes axis labels, a title and a legend, and is saved as a PNG file in `PLOT_DIR`.

In [11]:
def plot_timecourse(timecourses, variables):
    for var in variables:
        plt.figure(figsize=(7, 5))
        
        for name, result in timecourses.items():
            t_min = result['time'] / 60.0
            plt.plot(t_min, result[var], label=LABELS[name],
                      color=COLORS[name], linewidth=2)
            
        plt.axvline(T_DRUG / 60.0, color='black', linestyle='--', linewidth=1,
                    label='Cefoxitin added')
        
        plt.xlabel('time (min)')
        plt.ylabel(f'[{var}] (mM)')
        plt.title(f'{var} Concentration: WT vs ftsZ84, 30->42 C shift + cefoxitin')
        plt.legend(fontsize=8, loc='best')
        plt.tight_layout()
        
        filename = os.path.join(PLOT_DIR, var + ".png")
        plt.savefig(filename, dpi=150)
        
        plt.close()
        
        print(f"Saved {filename}")

## Fold-Change Comparison Generator

This function generates a grouped bar chart comparing the model-predicted AmpC fold induction with the literature-reported values from Uehara and Park (2002) for wild-type and ftsZ84 conditions at the selected time points (30, 60 and 90). The plot is saved as a PNG file in `PLOT_DIR`.

In [12]:
def plot_fold_comparison(fold_summary):    
    labels = ['WT_30', 'ftsZ84_30', 'WT_60', 'ftsZ84_60', 'WT_90', 'ftsZ84_90'] # labels used in the plot
    
    # Model fold-change values
    model_vals = [
    fold_summary.loc[fold_summary['Metric'] == 'Fold induction, WT', 'Model_30'].iloc[0],
    fold_summary.loc[fold_summary['Metric'] == 'Fold induction, ftsZ84', 'Model_30'].iloc[0],
    fold_summary.loc[fold_summary['Metric'] == 'Fold induction, WT', 'Model_60'].iloc[0],
    fold_summary.loc[fold_summary['Metric'] == 'Fold induction, ftsZ84', 'Model_60'].iloc[0],
    fold_summary.loc[fold_summary['Metric'] == 'Fold induction, WT', 'Model_90'].iloc[0],
    fold_summary.loc[fold_summary['Metric'] == 'Fold induction, ftsZ84', 'Model_90'].iloc[0]
    ]    

    # Literature fold-change values
    lit_vals = [
        LIT_TARGETS_FOLD["FOLD_WT_TARGET_30"],
        LIT_TARGETS_FOLD["FOLD_FTSZ_TARGET_30"],
        LIT_TARGETS_FOLD["FOLD_WT_TARGET_60"],
        LIT_TARGETS_FOLD["FOLD_FTSZ_TARGET_60"],
        LIT_TARGETS_FOLD["FOLD_WT_TARGET_90"],
        LIT_TARGETS_FOLD["FOLD_FTSZ_TARGET_90"]
    ]
 
    x = np.arange(len(labels))
    width = 0.35
 
    plt.figure(figsize=(6, 5))
    plt.bar(x - width / 2, model_vals, width, label='Model', color='#185FA5')
    plt.bar(x + width / 2, lit_vals, width, label='Uehara & Park 2002',
            color='#888780')
    
    plt.xticks(x, labels)
    plt.ylabel('AmpC Fold induction (+BL / -BL)')
    plt.title('Model vs. Literature AmpC Fold Induction by genotype')
    plt.legend()
    plt.tight_layout()
    
    filename = os.path.join(PLOT_DIR, "AmpC_fold_comparison.png")
    plt.savefig(filename, dpi=150)
    
    plt.close()
    
    print(f"Saved {filename}")

# Main

Main execution block.

In [13]:
if __name__ == '__main__':
    # Running the simulation experiment and storing results
    timecourses, sample_points, endpoint = run_uehara_park_validation()

    # Computing necessary metrics and storing results
    summary, fold_summary = compute_metrics(sample_points)

    # Printing summary
    print("\n--- AmpC at t = 3600 s vs. Table 2 raw values ---")
    print(summary.to_string(index=False))

    # Printing fold_summary
    print("\n--- Fold-induction and rho vs. Uehara & Park (2002) targets ---")
    print(fold_summary.to_string(index=False))

    # Saving summary and fold_summary as csv files
    summary.to_csv('uehara_park_ampc_summary.csv', index=False)
    fold_summary.to_csv('uehara_park_fold_summary.csv', index=False)

    # Specifying variables to plot
    variables = ['AmpC', 'mRNA_AmpC', 'BL', 'Z', 'FtsZ', 'AmpR', 'Anh_P', 'UDP_NAM_penta', 'Disacch_P_cyto', 'PG_UP', 'PG_UT', 'PG_XL', 'XL_f', 'Und_P_peri', 'PBP1a', 'PBP4', 'Disacch_P_peri', 'Lipid_II_peri', 'PBP1b', 'Und_PP_peri', 'Und_P_cyto']

    # Plotting timecourse of speficied variables
    plot_timecourse(timecourses, variables)

    # Plotting fold comparison bar chart
    plot_fold_comparison(fold_summary)

Shared burn-in converged via 'integration_fallback', residual=0.00e+00

--- AmpC at t = 3600 s vs. Table 2 raw values ---
         Condition  Model_AmpC_mM_30  Lit_activity_umol_mg_min_30  Model_AmpC_mM_60  Lit_activity_umol_mg_min_60  Model_AmpC_mM_90  Lit_activity_umol_mg_min_90
    WT, noninduced          0.043161                          0.5          0.043161                         0.40          0.043161                         0.50
       WT, induced          0.247674                          3.1          0.761217                         2.35          0.881796                         1.65
ftsZ84, noninduced          0.039651                          0.5          0.033063                         0.60          0.027374                         0.55
   ftsZ84, induced          0.055952                          0.7          0.085282                         0.85          0.091291                         0.55

--- Fold-induction and rho vs. Uehara & Park (2002) targets ---
             